# V7.3 — frozen READ on the matched metric

This firewalled stage reads only the sanitized clean manifest. It calls the unchanged 16-step `READ_IG` and `READ_local` implementation with identical answer-token IDs in engine and dashboard, and withholds the condition comparison until v7.4.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['PATH'] = '/home/jovyan/.local/bin:/home/jovyan/.npm-global/bin:' + os.environ['PATH']
os.environ['HF_HOME'] = '/home/jovyan/.cache/huggingface'
os.environ['HF_HUB_CACHE'] = '/home/jovyan/.cache/huggingface/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/home/jovyan/.cache/huggingface/hub'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print(ROOT)

/home/jovyan/j-space-thoughts


In [2]:
from src.matched_eval_v7 import run_read_stage_v7

def progress(completed, total, row):
    if completed == 1 or completed % 10 == 0 or completed == total:
        print(f'{completed}/{total}: {dict(row)}', flush=True)

summary = run_read_stage_v7(progress=progress)
summary

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at ../aten/src/ATen/native/transformers/cuda/attention_backward.cu:102.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


1/70: {'pair_id': 'symmetric-000', 'status': 'OK', 'same_metric_in_both_conditions': True}


/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at ../aten/src/ATen/native/transformers/cuda/attention_backward.cu:655.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


10/70: {'pair_id': 'symmetric-017', 'status': 'OK', 'same_metric_in_both_conditions': True}


...

40/70: {'pair_id': 'symmetric-053', 'status': 'OK', 'same_metric_in_both_conditions': True}


50/70: {'pair_id': 'symmetric-081', 'status': 'OK', 'same_metric_in_both_conditions': True}


60/70: {'pair_id': 'symmetric-095', 'status': 'OK', 'same_metric_in_both_conditions': True}


70/70: {'pair_id': 'symmetric-105', 'status': 'OK', 'same_metric_in_both_conditions': True}


{'read_path': '/home/jovyan/j-space-thoughts/results/v7/raw/read_v7.json',
 'read_sha256': '7b8ebbfea655d8bc0082a6ee74bc308e37331369464c621c609917bdedb3c0e5',
 'n_pairs': 70,
 'firewall': {'status': 'PASS',
  'imports_by_file': {'src/cheap_read.py': ['__future__',
    '__future__.annotations',
    'collections.abc',
    'collections.abc.Callable',
    'collections.abc.Sequence',
    'torch',
    'torch',
    'torch.nn',
    'typing',
    'typing.Any'],
   'src/matched_eval_v7.py': ['__future__',
    '__future__.annotations',
    'ast',
    'collections.abc',
    'collections.abc.Callable',
    'collections.abc.Mapping',
    'collections.abc.Sequence',
    'hashlib',
    'json',
    'pathlib',
    'pathlib.Path',
    'src.cheap_read',
    'src.cheap_read.score_prompt_pair',
    'src.cheap_read.weight_norm_capacity_baseline',
    'src.datasets',
    'src.datasets.validate_sanitized_manifest',
    'src.jlens_interface',
    'src.jlens_interface.MODEL_ID',
    'src.jlens_interface.MODEL_RE

In [3]:
import json
import subprocess

artifact = json.loads(Path(summary['read_path']).read_text())
assert artifact['ig_steps'] == 16
assert artifact['firewall']['status'] == 'PASS'
assert artifact['firewall']['causal_artifact_read'] is False
assert artifact['firewall']['edited_metrics_read'] is False
assert artifact['firewall']['causal_outputs_consumed'] is False
assert artifact['metric_contract']['engine'] == artifact['metric_contract']['dashboard']
assert all(row['same_metric_in_both_conditions'] for row in artifact['rows'])
assert all(row['verification_status'] == 'VERIFIED' for row in artifact['rows'])
assert all(row['engine']['ig_steps'] == row['dashboard']['ig_steps'] == 16 for row in artifact['rows'])
assert all(row['engine']['causal_outputs_consumed'] is False for row in artifact['rows'])
assert all(row['dashboard']['causal_outputs_consumed'] is False for row in artifact['rows'])

grep = subprocess.run(
    [
        'grep', '-nEi',
        r'^(from|import)[[:space:]].*(causal|interchange|intervention|patch|read_scores|read_validation)',
        'src/cheap_read.py', 'src/matched_eval_v7.py',
        'src/datasets.py', 'src/jlens_interface.py',
    ],
    text=True, capture_output=True, check=False,
)
assert grep.returncode == 1, grep.stdout + grep.stderr
{
    'n_pairs': len(artifact['rows']),
    'execution': artifact['execution'],
    'firewall': artifact['firewall'],
    'grep_forbidden_imports': grep.stdout,
    'condition_comparison_withheld_until_v7_4': True,
}

{'n_pairs': 70,
 'execution': {'capacity_baseline_pair_median': 0.03163323365151882,
  'condition_comparison_withheld_until_v7_4': True,
  'dashboard_read_batch_top1_consistent': 70,
  'engine_read_batch_top1_consistent': 70,
  'finite_READ_IG': True,
  'finite_READ_local': True,
  'ig_steps': 16,
  'n_dependency_groups': 24,
  'n_pairs': 70},
 'firewall': {'causal_artifact_read': False,
  'causal_outputs_consumed': False,
  'edited_metrics_read': False,
  'forbidden_import_fragments': ['causal',
   'interchange',
   'intervention',
   'patch',
   'read_scores',
   'read_validation'],
  'forbidden_imports_found': [],
  'frozen_read_sha256_after': 'a4a0ab35c50ce73dd153414118e6150891a708acf5f64bf9c8cb31225bb0caab',
  'frozen_read_sha256_before': 'a4a0ab35c50ce73dd153414118e6150891a708acf5f64bf9c8cb31225bb0caab',
  'imports_by_file': {'src/cheap_read.py': ['__future__',
    '__future__.annotations',
    'collections.abc',
    'collections.abc.Callable',
    'collections.abc.Sequence',
   